# Load two best model and test data from the web

I've got several models that showed good performance on test data. However, if I discard the ones that also showed strong overfitting for training data, I can showcase only one with specific depth. All other will be around the same, so I'll just choose one of them for comparison, which was somewhat good.

In [ ]:
import tensorflow as tf

In [ ]:
base_url = 'https://github.com/avasenin-14/applied-deep-learning/raw/main/'

model_act_linear_url = base_url + 'model_act_linear.keras'
model_depth_2_url = base_url + 'model_depth_2.keras'
test_csv_url = base_url + 'test.csv'

model_act_linear_path = tf.keras.utils.get_file('model_act_linear.keras', model_act_linear_url)
model_depth_2_path = tf.keras.utils.get_file('model_depth_2.keras', model_depth_2_url)
test_csv_path = tf.keras.utils.get_file('test.csv', test_csv_url)

print(f"'model_act_linear.keras' downloaded to: {model_act_linear_path}")
print(f"'model_depth_2.keras' downloaded to: {model_depth_2_path}")
print(f"'test.csv' downloaded to: {test_csv_path}")

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("frabbisw/facial-age")

print("Path to dataset files:", dataset_path)

In [ ]:
model_act_linear = tf.keras.models.load_model(model_act_linear_path)
model_depth_2 = tf.keras.models.load_model(model_depth_2_path)

print("Models 'model_act_linear' and 'model_depth_2' loaded successfully.")

In [ ]:
import pandas as pd
import numpy as np

test_df = pd.read_csv(test_csv_path)
print("test.csv loaded into DataFrame successfully.")
print(test_df.head())

# The 'file' column already contains the full path to each image.
test_df['full_image_path'] = test_df['file']

print("Full image paths created successfully.")
print(test_df[['file', 'full_image_path']].head())

In [ ]:
import tensorflow as tf
import numpy as np

# Initialize empty lists for images and labels with new naming convention
preprocessed_images_64x64 = []
age_bin_labels_64x64 = []

# Define target image size as 64x64
IMG_HEIGHT = 64
IMG_WIDTH = 64

# Loop through each row of the test_df to load and preprocess images
for index, row in test_df.iterrows():
    img_path = row['full_image_path']
    label = row['age_bins']

    # Load the image
    img = tf.keras.utils.load_img(img_path)

    # Convert image to NumPy array
    img_array = tf.keras.utils.img_to_array(img)

    # Resize image to 64x64
    img_resized = tf.image.resize(img_array, (IMG_HEIGHT, IMG_WIDTH))

    # Normalize pixel values to [0, 1]
    img_normalized = img_resized / 255.0

    # Append to lists
    preprocessed_images_64x64.append(img_normalized)
    age_bin_labels_64x64.append(label)

# Convert lists to NumPy arrays
X_test_64x64 = np.array(preprocessed_images_64x64)
y_test_age_bins_64x64 = np.array(age_bin_labels_64x64)

print("Preprocessed images (64x64) and age bin labels created successfully.")
print(f"Shape of preprocessed images (X_test_64x64): {X_test_64x64.shape}")
print(f"Shape of age bin labels (y_test_age_bins_64x64): {y_test_age_bins_64x64.shape}")

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Ensure `label_to_index` is consistent or create one if not present.
# Assuming `label_to_index` is available from previous steps and represents all possible age bins.
# If `label_to_index` is not directly suitable for LabelEncoder, rebuild it from unique sorted categories.

# Create a LabelEncoder and fit it with the keys from label_to_index to maintain consistency
le = LabelEncoder()
# Fit to the unique values that the model was trained on, assumed to be represented by label_to_index keys
le.fit(list(label_to_index.keys()))

# Convert age bin labels to numerical format using the fitted LabelEncoder
y_test_numerical_64x64 = le.transform(y_test_age_bins_64x64)

# One-hot encode the numerical labels
# `num_classes` is already available from kernel state, representing the total number of age bins
y_test_encoded_64x64 = tf.keras.utils.to_categorical(y_test_numerical_64x64, num_classes=num_classes)

print(f"Shape of y_test_numerical_64x64: {y_test_numerical_64x64.shape}")
print(f"Shape of y_test_encoded_64x64: {y_test_encoded_64x64.shape}")
print("Labels converted to numerical and one-hot encoded successfully.")

# Showcase models performance

In [ ]:
import tensorflow as tf

# Reload the models to ensure a clean state for evaluation
model_act_linear = tf.keras.models.load_model(model_act_linear_path)
model_depth_2 = tf.keras.models.load_model(model_depth_2_path)

print("Evaluating model_act_linear...")
loss_act_linear, accuracy_act_linear = model_act_linear.evaluate(X_test_64x64, y_test_numerical_64x64)
print(f"model_act_linear - Loss: {loss_act_linear:.4f}, Accuracy: {accuracy_act_linear:.4f}")

print("\nEvaluating model_depth_2...")
loss_depth_2, accuracy_depth_2 = model_depth_2.evaluate(X_test_64x64, y_test_numerical_64x64)
print(f"model_depth_2 - Loss: {loss_depth_2:.4f}, Accuracy: {accuracy_depth_2:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare data for plotting
metrics = {
    'Model': ['model_act_linear', 'model_depth_2'],
    'Loss': [loss_act_linear, loss_depth_2],
    'Accuracy': [accuracy_act_linear, accuracy_depth_2]
}
metrics_df = pd.DataFrame(metrics)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Loss
sns.barplot(x='Model', y='Loss', data=metrics_df, ax=axes[0])
axes[0].set_title('Model Loss Comparison')
axes[0].set_ylabel('Loss')
axes[0].set_ylim(0, max(metrics_df['Loss']) * 1.2)
for index, row in metrics_df.iterrows():
    axes[0].text(index, row['Loss'] + 0.02, f"{row['Loss']:.4f}", color='black', ha="center")

# Plot Accuracy
sns.barplot(x='Model', y='Accuracy', data=metrics_df, ax=axes[1])
axes[1].set_title('Model Accuracy Comparison')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.0)
for index, row in metrics_df.iterrows():
    axes[1].text(index, row['Accuracy'] + 0.02, f"{row['Accuracy']:.4f}", color='black', ha="center")

plt.tight_layout()
plt.show()

# Conclusion
As we can see model with specific depth (it was 2 * (dense layer + pooling) + flattening + dense + final layer) achieves better accuracy (0.8252) and loss (0.5728) results, and the same time not falling to overfitting.
The other model with linear final layer activation function shows slightly worse results for accuracy (0.6728) and loss (0.8185) metrics. Nevertheless, it is still a good model, and showed better results than transfer learning on a custom encoder or general purpose image model.